第一步：导入相关包

In [1]:
# 基础文件处理
from PyPDF2 import PdfReader
# 文本分割
from langchain.text_splitter import RecursiveCharacterTextSplitter
# 嵌入模型（阿里云DashScope）
from langchain_community.embeddings import DashScopeEmbeddings
# 向量数据库（FAISS）
from langchain_community.vectorstores import FAISS
# 问答链
from langchain.chains.question_answering import load_qa_chain
# 大模型（通义千问）
from langchain_community.llms import Tongyi
# 成本统计
from langchain_community.callbacks import get_openai_callback

# 配置阿里云灵积API Key（替换为你的有效Key）
DASHSCOPE_API_KEY = "sk-bb5a3bc665be4d35bea3efce0196c8c1"

第二步：PDF 文本提取

In [2]:
# -------------------------- 步骤2：PDF文本提取 --------------------------
# 1. 读取PDF文件
pdf_path = r"C:\Users\23017\Desktop\LLM\RAG\RAG_case\浦发上海浦东发展银行西安分行个金客户经理考核办法.pdf"
pdf_reader = PdfReader(pdf_path)
print(f"PDF文件总页数：{len(pdf_reader.pages)}")

# 2. 逐页提取文本，记录每页文本和对应页码
page_texts = []  # 存储每页的完整文本
page_numbers = []  # 存储每页对应的页码

for page_idx, page in enumerate(pdf_reader.pages, start=1):
    # 提取单页文本
    extracted_text = page.extract_text()
    if extracted_text:
        clean_text = extracted_text.strip()  # 清理首尾空格
        page_texts.append(clean_text)
        page_numbers.append(page_idx)
        print(f"第{page_idx}页提取完成，文本长度：{len(clean_text)}字符")
    else:
        print(f"第{page_idx}页无有效文本")

# 打印提取结果预览
print(f"\n共提取到 {len(page_texts)} 页有效文本")
print(f"第1页文本预览：{page_texts[0][:200]}...")


PDF文件总页数：9
第1页提取完成，文本长度：438字符
第2页提取完成，文本长度：381字符
第3页提取完成，文本长度：434字符
第4页提取完成，文本长度：420字符
第5页提取完成，文本长度：537字符
第6页提取完成，文本长度：511字符
第7页提取完成，文本长度：567字符
第8页提取完成，文本长度：474字符
第9页提取完成，文本长度：93字符

共提取到 9 页有效文本
第1页文本预览：百度文库  - 好好学习，天天向上  
-1 上海浦东发展银行西安分行  
个金客户经理管理考核暂行办法  
 
 
第一章  总   则 
第一条   为保证我分行个金客户经理制的顺利实施，有效调动个
金客户经理的积极性，促进个金业务快速、稳定地发展，根据总行《上
海浦东发展银行个人金融营销体系建设方案（试行）》要求，特制定
《上海浦东发展银行西安分行个金客户经理管理考核暂行办法（试
行）》（以...


第三步：文本分割

In [3]:
# -------------------------- 步骤3：文本分割 --------------------------
# 1. 初始化中文友好的文本分割器
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", "。", "！", "？", "，", "、", "."],  # 优先按中文分句
    chunk_size=1000,        # 每个文本块最大字符数
    chunk_overlap=200,      # 文本块重叠字符（保证上下文连贯）
    length_function=len,    # 按字符长度分割
)

# 2. 分割文本并绑定每个文本块的页码
all_chunks = []               # 存储所有分割后的文本块
chunk_page_mapping = {}       # 关键：文本块 → 对应页码的映射字典

for text, page_num in zip(page_texts, page_numbers):
    # 分割单页文本为多个小块
    chunks = text_splitter.split_text(text)
    for chunk in chunks:
        chunk = chunk.strip()
        if chunk:  # 跳过空文本块
            all_chunks.append(chunk)
            chunk_page_mapping[chunk] = page_num  # 记录每个块的页码

# 打印分割结果
print(f"\n文本分割完成：")
print(f"- 原始有效页数：{len(page_texts)}")
print(f"- 分割后文本块数量：{len(all_chunks)}")
print(f"- 第1个文本块预览：{all_chunks[0][:100]}... 对应页码：{chunk_page_mapping[all_chunks[0]]}")


文本分割完成：
- 原始有效页数：9
- 分割后文本块数量：9
- 第1个文本块预览：百度文库  - 好好学习，天天向上  
-1 上海浦东发展银行西安分行  
个金客户经理管理考核暂行办法  
 
 
第一章  总   则 
第一条   为保证我分行个金客户经理制的顺利实施，有效调动... 对应页码：1


第四步：Chroma向量库

In [4]:
# -------------------------- 步骤4：Chroma向量库 --------------------------
# 替换FAISS为Chroma（需先安装：pip install chromadb）
from langchain_community.vectorstores import Chroma

# 初始化嵌入模型（保持不变）
embeddings = DashScopeEmbeddings(
    model="text-embedding-v1",
    dashscope_api_key=DASHSCOPE_API_KEY,
)

# 构建Chroma向量库（替代FAISS）
knowledge_base = Chroma.from_texts(
    texts=all_chunks,       # 分割后的文本块
    embedding=embeddings,   # 嵌入模型
    persist_directory="./local_rag_chroma_db"  # 本地保存路径
)

# 绑定页码映射（保持不变）
knowledge_base.chunk_page_mapping = chunk_page_mapping

# 持久化向量库（Chroma需手动执行persist）
knowledge_base.persist()
print(f"\n Chroma向量库已保存至本地：.RAG_case/local_rag_chroma_db_case")




 Chroma向量库已保存至本地：.RAG_case/local_rag_chroma_db_case


C:\Users\23017\AppData\Local\Temp\ipykernel_39324\2139418100.py:22: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  knowledge_base.persist()


第五步：问答查询

In [5]:
# -------------------------- 步骤5：问答查询 --------------------------
# 1. 初始化通义千问大模型
llm = Tongyi(
    model_name='qwen-turbo',        # 通义千问轻量版（速度快、成本低）
    dashscope_api_key=DASHSCOPE_API_KEY,
    temperature=0.1,               # 降低随机性，保证回答准确性
)

# 2. 设置查询问题（可替换为任意问题）
query = "客户经理被投诉了，投诉一次扣多少分"
print(f"\n 开始查询：{query}")

# 3. 相似度检索（从向量库找最相关的文本块）
top_k = 3  # 返回最相关的3个文本块
related_docs = knowledge_base.similarity_search(query, k=top_k)
print(f"📄 检索到 {len(related_docs)} 个相关文本块")

# 4. 加载问答链（将检索结果拼接后传给LLM生成回答）
qa_chain = load_qa_chain(
    llm=llm,
    chain_type="stuff"  # 适合短文本：将所有相关文本块拼接成prompt
)

# 5. 执行问答并统计API成本
with get_openai_callback() as cost:
    # 执行问答
    response = qa_chain.invoke({
        "input_documents": related_docs,  # 检索到的相关文本
        "question": query                # 用户问题
    })

# 6. 打印结果
print("\n===== 问答结果 =====")
print(f"问题：{query}")
print(f"回答：{response['output_text']}")
print(f"\n API调用成本：{cost}")

# 7. 打印回答来源页码（去重）
print("\n===== 回答来源页码 =====")
unique_pages = set()
for doc in related_docs:
    chunk_content = doc.page_content.strip()
    # 从映射中获取文本块对应的页码
    source_page = knowledge_base.chunk_page_mapping.get(chunk_content, "未知")
    if source_page not in unique_pages:
        unique_pages.add(source_page)
        print(f"- 文本块来源页码：{source_page}")

if not unique_pages:
    print(" 未找到相关文本块的页码信息")


 开始查询：客户经理被投诉了，投诉一次扣多少分
📄 检索到 3 个相关文本块


C:\Users\23017\AppData\Local\Temp\ipykernel_39324\260125483.py:19: LangChainDeprecationWarning: This class is deprecated. See the following migration guides for replacements based on `chain_type`:
stuff: https://python.langchain.com/docs/versions/migrating_chains/stuff_docs_chain
map_reduce: https://python.langchain.com/docs/versions/migrating_chains/map_reduce_chain
refine: https://python.langchain.com/docs/versions/migrating_chains/refine_chain
map_rerank: https://python.langchain.com/docs/versions/migrating_chains/map_rerank_docs_chain

See also guides on retrieval and question-answering here: https://python.langchain.com/docs/how_to/#qa-with-rag
  qa_chain = load_qa_chain(



===== 问答结果 =====
问题：客户经理被投诉了，投诉一次扣多少分
回答：客户经理被投诉一次扣2分。

 API调用成本：Tokens Used: 0
	Prompt Tokens: 0
		Prompt Tokens Cached: 0
	Completion Tokens: 0
		Reasoning Tokens: 0
Successful Requests: 1
Total Cost (USD): $0.0

===== 回答来源页码 =====
- 文本块来源页码：5
- 文本块来源页码：8
- 文本块来源页码：6
